In [ ]:
The marketing team reports that the user engagement metrics code is showing impossible results (bounce rates over 100%, negative session times).

Problematic Code:

def analyze_user_engagement(logs_df):
    metrics = {
        'bounce_rate': logs_df.groupby('session_id')['page_views'].apply(
            lambda x: x == 1).mean(),
        'avg_session_time': logs_df.groupby('session_id')['duration'].sum(),
        'pages_per_session': logs_df.groupby('session_id')['page_views'].mean()
    }
    
    device_metrics = logs_df.groupby('device_type').agg({
        'session_id': 'count',
        'duration': 'mean',
        'page_views': 'sum'
    })
    
    return metrics, device_metrics
Task:

Debug the calculations.
Add data validation.
Implement proper time calculations.
Create summary visualizations.

In [ ]:
The issue comes from **incorrect calculations and missing validation**:

### Problems in the Original Code

1. **Bounce rate calculation is wrong**

   * `lambda x: x == 1` returns True/False for every row, not per session.
   * Bounce rate should be:

[
\text{Bounce Rate} =
\frac{\text{Sessions with exactly 1 page view}}
{\text{Total Sessions}}
\times 100
]

2. **Negative session times**

   * Caused by invalid `duration` values.
   * We need validation to remove/fix negative durations.

3. **Average session time**

   * Original code returns all session sums instead of the average.

4. **Pages per session**

   * Mean page views per row is incorrect.
   * We need **total pages per session**, then average.

---

## Corrected Code with Validation + Visualizations

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def analyze_user_engagement(logs_df):
    """
    Analyze user engagement metrics safely
    """

    try:
        # -----------------------------
        # DATA VALIDATION
        # -----------------------------

        required_columns = [
            'session_id',
            'page_views',
            'duration',
            'device_type'
        ]

        # Check missing columns
        missing_cols = [
            col for col in required_columns
            if col not in logs_df.columns
        ]

        if missing_cols:
            raise ValueError(
                f"Missing columns: {missing_cols}"
            )

        # Remove duplicates
        logs_df = logs_df.drop_duplicates()

        # Remove negative durations
        logs_df = logs_df[
            logs_df['duration'] >= 0
        ]

        # Remove invalid page views
        logs_df = logs_df[
            logs_df['page_views'] >= 0
        ]

        # Handle missing values
        logs_df = logs_df.dropna()

        # -----------------------------
        # SESSION-LEVEL CALCULATIONS
        # -----------------------------

        session_metrics = logs_df.groupby(
            'session_id'
        ).agg({
            'page_views': 'sum',
            'duration': 'sum'
        }).reset_index()

        # -----------------------------
        # BOUNCE RATE
        # Session with exactly 1 page
        # -----------------------------

        bounced_sessions = (
            session_metrics['page_views'] == 1
        ).sum()

        total_sessions = (
            session_metrics['session_id']
            .nunique()
        )

        bounce_rate = (
            bounced_sessions /
            total_sessions
        ) * 100

        # -----------------------------
        # AVG SESSION TIME
        # -----------------------------

        avg_session_time = (
            session_metrics['duration']
            .mean()
        )

        # -----------------------------
        # PAGES PER SESSION
        # -----------------------------

        pages_per_session = (
            session_metrics['page_views']
            .mean()
        )

        metrics = {
            'bounce_rate (%)':
                round(bounce_rate, 2),

            'avg_session_time (sec)':
                round(avg_session_time, 2),

            'pages_per_session':
                round(pages_per_session, 2)
        }

        # -----------------------------
        # DEVICE METRICS
        # -----------------------------

        device_metrics = logs_df.groupby(
            'device_type'
        ).agg(
            sessions=('session_id', 'nunique'),
            avg_duration=('duration', 'mean'),
            total_page_views=('page_views', 'sum')
        )

        # -----------------------------
        # VISUALIZATIONS
        # -----------------------------

        # 1. Device Sessions
        plt.figure(figsize=(8, 5))
        device_metrics['sessions'].plot(
            kind='bar'
        )

        plt.title(
            'Sessions by Device Type'
        )
        plt.xlabel('Device Type')
        plt.ylabel('Number of Sessions')
        plt.show()

        # 2. Average Duration
        plt.figure(figsize=(8, 5))
        device_metrics['avg_duration'].plot(
            kind='bar'
        )

        plt.title(
            'Average Session Duration by Device'
        )
        plt.xlabel('Device Type')
        plt.ylabel('Average Duration (sec)')
        plt.show()

        # 3. Session Duration Distribution
        plt.figure(figsize=(8, 5))
        plt.hist(
            session_metrics['duration'],
            bins=10
        )

        plt.title(
            'Session Duration Distribution'
        )
        plt.xlabel('Duration (sec)')
        plt.ylabel('Frequency')
        plt.show()

        return metrics, device_metrics

    except Exception as e:
        print(
            f"Error analyzing engagement: {e}"
        )
        return None, None
```

## Why These Visualizations?

### 1. Bar Chart — Sessions by Device

**Purpose:** Compare engagement across device types.

Shows:

* Mobile vs desktop traffic
* Which device generates more sessions

---

### 2. Bar Chart — Average Session Duration

**Purpose:** Compare time spent per device.

Shows:

* Which devices have longer engagement
* Possible UX issues on certain devices

Example:

* Mobile users spending less time may signal poor mobile experience.

---

### 3. Histogram — Session Duration Distribution

**Purpose:** Understand session behavior patterns.

Shows:

* Most common session lengths
* Outliers or abnormal durations

Example:

* Many short sessions = high disengagement.
* Long sessions = stronger engagement.

### Key Fixes Implemented

✔ Bounce rate capped properly (0–100%)
✔ Negative durations removed
✔ Proper session aggregation
✔ Missing column checks
✔ Duplicate removal
✔ Correct time calculations
✔ Visual summaries for decision-making